# 🎯 LSH + MinHash (Recomendaciones) — Netflix EDA

**Sistema de Recomendación por Co-Visionado**  
Curso: Estructuras de Datos y Algoritmos | UTEC

---

## Fundamentos Matemáticos

### Similitud de Jaccard

Mide qué tan similares son dos conjuntos:
$$J(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

Para videos: Si A = usuarios que vieron "Stranger Things" y B = usuarios que vieron "Dark", J(A,B) alta → videos similares → buenos para recomendar juntos.

### MinHash

Estima la similitud de Jaccard sin comparar los conjuntos completos. La idea clave:
> **La probabilidad de que h_min(A) = h_min(B) es exactamente J(A, B)**

Usando `k` funciones hash independientes, se obtiene una firma de `k` valores que permite estimar J con error ≈ 1/√k.

### LSH (Locality Sensitive Hashing)

Permite encontrar candidatos similares sin comparar todos los pares O(n²):
1. Dividir la firma en `b` bandas de `r` filas cada una
2. Dos videos son candidatos si **al menos una banda** tiene la misma sub-firma
3. Probabilidad de ser candidatos con similitud s: P = 1 - (1 - s^r)^b

### Uso en Netflix

Con 100M+ videos, comparar todos los pares sería O(n²) = 10¹⁶ operaciones. LSH lo reduce a ~O(n).

In [ ]:
import sys, os
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !git clone https://github.com/Guido-Silva/netflix-streaming-eda.git
    %cd netflix-streaming-eda
    sys.path.insert(0, '.')
else:
    sys.path.insert(0, os.path.dirname(os.getcwd()))
print('✅ Entorno configurado')

##Rubén Git

In [ ]:
from src.lsh_minhash import LSHMinHash, MinHash
print('✅ Módulo LSH + MinHash importado')

In [ ]:
from src.lsh_minhash import demo
demo()

In [ ]:
# Experimento: Precisión de MinHash vs número de hashes
import random
import numpy as np

print('📊 Experimento: Precisión de MinHash vs Número de Hashes')
print('=' * 60)

random.seed(42)
N_USUARIOS = 500
todos_usuarios = [f'u{i}' for i in range(N_USUARIOS)]

# Crear pares con similitudes conocidas
similitudes_objetivo = [0.1, 0.3, 0.5, 0.7, 0.9]
num_hashes_list = [10, 20, 50, 100, 200]

# Para cada similitud objetivo, crear dos conjuntos
pares = []
for sim_target in similitudes_objetivo:
    n1 = 100
    n_comun = int(n1 * sim_target / (2 - sim_target))
    n_comun = min(n_comun, n1)

    base = set(random.sample(todos_usuarios, n1))
    otros = [u for u in todos_usuarios if u not in base]

    set1 = base
    set2 = set(random.sample(list(base), n_comun)) | set(random.sample(otros, n1 - n_comun))

    lsh_temp = LSHMinHash(num_hashes=10, num_bands=2)
    sim_real = lsh_temp.jaccard_similarity(set1, set2)
    pares.append((set1, set2, sim_real))

# Medir error para cada número de hashes
errores_por_hashes = []
for num_h in num_hashes_list:
    minhash = MinHash(num_hashes=num_h)
    errores = []
    for set1, set2, sim_real in pares:
        f1 = minhash.compute_signature(set1)
        f2 = minhash.compute_signature(set2)
        sim_est = minhash.similarity(f1, f2)
        errores.append(abs(sim_est - sim_real))
    error_prom = np.mean(errores)
    errores_por_hashes.append(error_prom)
    print(f'  num_hashes={num_h:>4}: error_promedio={error_prom:.4f} (teórico: 1/sqrt(k)={1/num_h**0.5:.4f})')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Grafico 1: Error vs Número de Hashes
axes[0].plot(num_hashes_list, errores_por_hashes, 'bo-', label='Error observado', linewidth=2, markersize=8)
ref_teorica = [1/k**0.5 for k in num_hashes_list]
axes[0].plot(num_hashes_list, ref_teorica, 'r--', label='1/√k (teórico)', linewidth=2)
axes[0].set_xlabel('Número de Hashes (k)')
axes[0].set_ylabel('Error Absoluto Promedio')
axes[0].set_title('Precisión de MinHash vs Número de Hashes')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Grafico 2: Matriz de similitud entre videos
random.seed(42)
lsh = LSHMinHash(num_hashes=100, num_bands=20)
videos_nombre = ['Stranger Things', 'Dark', 'Squid Game', 'Alice Borderland',
                 'La Casa Papel', 'Money Heist', 'The Crown', 'Bridgerton']
n_v = len(videos_nombre)
grupos_usuarios = [
    set(random.sample(todos_usuarios, 100)),
    set(random.sample(todos_usuarios, 100)),
    set(random.sample(todos_usuarios, 90)),
    set(random.sample(todos_usuarios, 90)),
    set(random.sample(todos_usuarios, 80)),
    set(random.sample(todos_usuarios, 80)),
    set(random.sample(todos_usuarios, 70)),
    set(random.sample(todos_usuarios, 70)),
]
# Hacer pares similares compartiendo usuarios
for i in [0,2,4,6]:
    base = grupos_usuarios[i]
    otros = [u for u in todos_usuarios if u not in base]
    comunes = set(random.sample(list(base), int(len(base)*0.7)))
    grupos_usuarios[i+1] = comunes | set(random.sample(otros, len(base) - len(comunes)))

vid_ids = [f'v{i}' for i in range(n_v)]
for vid_id, usuarios in zip(vid_ids, grupos_usuarios):
    lsh.add_video(vid_id, usuarios)

matriz_sim = np.zeros((n_v, n_v))
for i in range(n_v):
    for j in range(n_v):
        matriz_sim[i][j] = lsh.jaccard_similarity(grupos_usuarios[i], grupos_usuarios[j])

im = axes[1].imshow(matriz_sim, cmap='Blues', vmin=0, vmax=1)
axes[1].set_xticks(range(n_v))
axes[1].set_yticks(range(n_v))
axes[1].set_xticklabels([v[:10] for v in videos_nombre], rotation=45, ha='right')
axes[1].set_yticklabels([v[:10] for v in videos_nombre])
axes[1].set_title('Matriz de Similitud de Jaccard entre Videos')
plt.colorbar(im, ax=axes[1], label='Similitud Jaccard')

for i in range(n_v):
    for j in range(n_v):
        axes[1].text(j, i, f'{matriz_sim[i][j]:.2f}', ha='center', va='center',
                    color='black' if matriz_sim[i][j] < 0.5 else 'white', fontsize=7)

plt.suptitle('Análisis LSH + MinHash — Sistema de Recomendaciones Netflix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('lsh_minhash_analysis.png', dpi=100, bbox_inches='tight')
plt.show()

## 📊 Análisis de Complejidad

| Operación | Complejidad Temporal | Complejidad Espacial |
|-----------|---------------------|---------------------|
| `add_video(id, user_set)` | **O(\|user_set\| × h)** | O(h) por video |
| `find_similar(id, threshold)` | **O(b × r + \|candidatos\|)** | O(1) |
| `jaccard_similarity(A, B)` | **O(\|A\| + \|B\|)** | O(\|A\| + \|B\|) |

donde `h` = num_hashes, `b` = num_bands, `r` = rows_per_band.

### Ventaja sobre Comparación Exhaustiva

| Enfoque | Complejidad | Para 1M videos |
|---------|------------|----------------|
| Comparación par-a-par | O(n²) | 10¹² operaciones |
| **LSH + MinHash** | **O(n)** aprox | **~10⁶ operaciones** |

LSH es aproximadamente **1,000,000x más rápido** para 1M videos, con pequeña pérdida de precisión controlable con los parámetros `b` y `r`.